# TopiOCQA with PyTerrier-RAG

This notebook demonstrates access to the TopiOCQA conversational QA dataset through the PyTerrier datasets API.

The adapter keeps the current question in `query` and the preceding conversational context in `history`, allowing users to define their own conversational query formulation.

Dataset: https://huggingface.co/datasets/McGill-NLP/TopiOCQA  
Paper: https://aclanthology.org/2022.tacl-1.27/


In [ ]:
import pyterrier as pt
import pyterrier_rag


## Load the development split

`rag:topiocqa` exposes topics, gold answers, and qrels. The `dev` alias maps to the public TopiOCQA validation split.


In [ ]:
dataset = pt.get_dataset("rag:topiocqa")

topics = dataset.get_topics("dev")
answers = dataset.get_answers("dev")
qrels = dataset.get_qrels("dev")

print("topics:", topics.shape)
print("answers:", answers.shape)
print("qrels:", qrels.shape)
print("topic columns:", topics.columns.tolist())


## Inspect a conversational turn

Each `qid` is formed as `<conversation_id>_<turn_id>`. The current question is stored separately from its conversational history.


In [ ]:
qid = "100_2"

display(topics[topics["qid"] == qid])
display(answers[answers["qid"] == qid])
display(qrels[qrels["qid"] == qid])


## Define conversational query formulations

Because `history` is preserved as a separate field, the adapter does not impose a particular query-rewriting strategy. For example, we can compare the original current-turn query with a simple all-history concatenation.


In [ ]:
topics_original = topics.copy()

topics_all_history = topics.copy()
topics_all_history["query"] = topics_all_history.apply(
    lambda row: " ".join([*map(str, row["history"]), str(row["query"])]),
    axis=1,
)

example = topics_original[topics_original["qid"] == qid].iloc[0]
example_all_history = topics_all_history[topics_all_history["qid"] == qid].iloc[0]

print("Original query:")
print(example["query"])
print("\nHistory:")
print(example["history"])
print("\nAll-history query:")
print(example_all_history["query"])


The resulting topic DataFrames can be passed to downstream PyTerrier retrieval or RAG pipelines. Corpus/index selection is intentionally left to the user rather than being fixed by the TopiOCQA adapter.
